# Business Analytics of Amazon Customer Reviews****

import packages or functions

In [1]:

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import json #to read java format file
import seaborn as sns #packages to be installed
import numpy as np #array processing
import os
import nltk #tokenization
from nltk.stem import WordNetLemmatizer
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import gzip, json, itertools
from collections import Counter
from wordcloud import WordCloud
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
df = pd.read_csv("/kaggle/input/amazon-customer-review/Amazon_review_data.csv") #read and save the data in a dataframe

**view first 5 rows in the dataframe or table**

In [3]:
df.head(5) #view the first 5 rows in the table

,Unnamed: 0,asin,overall,summary,reviewText
0,0,0143026860,1.0,One Star,great
1,1,0143026860,4.0,... to reading about the Negro Baseball and th...,My husband wanted to reading about the Negro ...
2,2,0143026860,4.0,Worth the Read,"This book was very informative, covering all a..."
3,3,0143026860,5.0,Good Read,I am already a baseball fan and knew a bit abo...
4,4,0143026860,5.0,"More than facts, a good story read!",This was a good story of the Black leagues. I ...


names of columns in the amazon table 

In [4]:
df.columns #list the names of columns in the table

Index(['Unnamed: 0', 'asin', 'overall', 'summary', 'reviewText'], dtype='object')

# **Data Cleaning**# 

1. drop unwanted column(s)

In [5]:
df = df.drop(df.columns[0], axis = 1) #drop the first columns containing only index number

In [6]:
df.columns

Index(['asin', 'overall', 'summary', 'reviewText'], dtype='object')

* rename the overall column to ratings which represent customer ratings
* rename the asin column to product_id

In [7]:
df.rename(columns = {"overall":"ratings"}, inplace = True)#rename the overall column to ratings which represent customer ratings
df.rename(columns ={'asin':'product_id'}, inplace =True) #rename asin to product_id
df.columns #check if the columns is still there

Index(['product_id', 'ratings', 'summary', 'reviewText'], dtype='object')

2. Check for rows with missing values

In [8]:
number_of_null_Reviews = df['reviewText'].isnull().sum()
print(f"the total number of rows with zero reviews in the review column is: {number_of_null_Reviews}" )

number_of_empty_rows_in_rating_column = df['ratings'].isnull().sum()
print(f"the total number of empty ratings is: {number_of_empty_rows_in_rating_column}")

the total number of rows with zero reviews in the review column is: 10
the total number of empty ratings is: 0


we have 10 rows with no review from customers but all purchased products have been rated. Thus, we drop empty rows from the review columns; this will increase our sentiment analysis accuracy.

In [9]:
#drop empty rows
df = df.dropna(subset =['ratings', 'reviewText']) #drop empty rows


In [10]:
#make sure that table maintained its structure and empty rows are cleaned
number_of_empty_rows = df["reviewText"].isnull().sum()
print(f"number of empty rows is:{number_of_empty_rows}")
df.head(3)

number of empty rows is:0


,product_id,ratings,summary,reviewText
0,0143026860,1.0,One Star,great
1,0143026860,4.0,... to reading about the Negro Baseball and th...,My husband wanted to reading about the Negro ...
2,0143026860,4.0,Worth the Read,"This book was very informative, covering all a..."


next we'll remove duplicates product_id

In [11]:
number_of_duplicates = df.duplicated(subset =['product_id']).sum() #check for duplicates
print(f"the of duplicates are: {number_of_duplicates}")

the of duplicates are: 338365


In [12]:
df.drop_duplicates(subset = ['product_id'], inplace = True) #drop d

finding the number of unique products

In [13]:
number_of_unique_products = df['product_id'].nunique()
print(f"there {number_of_unique_products} number of unique products")

there 32571 number of unique products


check for basic information about the data using descriptive statistics

In [14]:
df.head(5)

,product_id,ratings,summary,reviewText
0,0143026860,1.0,One Star,great
7,014789302X,1.0,One Star,I didn't like this product it smudged all unde...
17,0992916305,5.0,Gorgeous!,"LOVE, LOVE, LOVE!! The movie was breathtaking ..."
23,1620213982,5.0,cute!,The product was cute and easy to use. It arri...
4802,162209798X,5.0,Square one for living well,As an aspiring actor and singer and too infreq...


In [15]:
print(df['ratings'].describe())

count    32571.000000
mean         4.014062
std          1.446663
min          1.000000
25%          3.000000
50%          5.000000
75%          5.000000
max          5.000000
Name: ratings, dtype: float64


there are 32571 unique products in the beauty category with a mean rating of 4, stan dev. of 1.4, min of 1 and max rating of 5 from customers

copy and save a copy of the clean data

In [16]:
selected_products = df.copy()